In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab04.ipynb")

# Lab 5:  Model Misspecification and Sampling
Welcome to the fifth Data 102 lab! 

This lab has three goals. You will:

1. Study the sensitivity of Bayesian estimators to errors in the chosen prior model. 

1. Fit the parameters of a prior by maximizing the evidence. 

1. Draw inferential conclusions by sampling. 

The code and responses you need to fill in are represented by `...`. There is additional documentation for each part as you go along. 


## Collaboration Policy
Data science is a collaborative activity. While you may talk with others about the labs, we ask that you **write your solutions individually and do not share your code with anyone other than your partner**. If you do discuss the assignments with people other than your partner please **include their names** in the cell below.

You can submit the lab in pairs (groups of two, no more than two). **If you choose to work in a pair, please make sure to add your group member on Pensieve for the Lab 04 code submission.**

## Submission to Pensieve

**Since this lab involves sampling, tests may take awhile to run. Please submit as early as possible, as last minute submissions may overwhelm DataHub, preventing yourself and others from submitting on-time.**

For full credit, this assignment should be completed and submitted before Wednesday, Sep 24th, 2025 at 5:00 PM PST.

## Collaborators
Write the names of your collaborators in this cell.

`<Collaborator Name> <Collaborator e-mail>`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import beta, binom
import itertools
from ipywidgets import interact, interactive


import hashlib
%matplotlib inline

sns.set(style="dark")
plt.style.use("ggplot")

import pymc as pm
import logging
logger = logging.getLogger('pymc')
logger.setLevel(logging.INFO)

# Question 1: Sensitivity to Prior Misspecification

This question is closely related to discussion 5. Please review that question before starting here. Note, since this problem reuses work from discussion 5, the questions on discussion 5 will be in scope for your second exam. 

In most Bayesian contexts we do not know our priors exactly. So, it is important to study how much our posterior inferences could vary if we vary our prior parameters, and how robust our inferential procedures are to errors in the prior (that is, to prior misspecification). 

Consider the setting you explored in discussion 5. You can observe a string of i.i.d.~random variables $\{X_i\}_{i=1}^n$ drawn normally with unknown mean $\Theta$ and known variance $\sigma_x^2$. To estimate the mean you adopt the Bayesian model:

$$\Theta \sim \text{Normal}(\mu_{\theta},\sigma_{\theta}^2), \quad \quad \{X_i\}_{i=1}^n |\Theta = \theta \underset{\text{i.i.d}}{\sim} \text{Normal}(\theta,\sigma_x^2)$$

Then, under your model:

$$\Theta|\{X_i = x_i\}_{i=1}^n \sim \text{Normal}\left( a, b \right) $$

where:

$$\begin{aligned}
    & a = (1 - \lambda) \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) + \lambda \mu_{\theta}, \\
    & \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) = \frac{1}{n} \sum_{i=1}^n x_i, \\
    & \lambda = \frac{1/\sigma_{\theta}^2}{n/\sigma_x^2 + 1/\sigma_{\theta}^2} \\
    & b = \frac{1}{n/\sigma_x^2 + 1/\sigma_{\theta}^2}
\end{aligned} $$

You adopt the posterior mean as your estimator:

$$\hat{\theta}_{\text{mean}}(\{x_i\}_{i=1}^n) = \mathbb{E}[\Theta|\{X_i = x_i\}_{i=1}^n] =  (1 - \lambda) \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) + \lambda \mu_{\theta}$$

Suppose that, in truth:

$$\Theta \sim \text{Normal}(0,1). $$

Then, your model is misspecified whenever $\mu_{\theta} \neq 0$ and $\sigma_{\theta} \neq 1$. Let's work out how much error we can admit before our posterior mean estimator is less reliable (has a larger risk) than the MLE estimator which did not require a prior. 

## 1a) Checking the MLE

Confirm that, the MLE is the sample average, $\bar{x}_n$:
    
$$\hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) = \bar{x}_n = \frac{1}{n} \sum_{i=1}^n x_i. $$

_Type your answer here, replacing this text._

## 1b) Generality

It may seem odd that we are starting from a fixed true model. Show that, if, in truth, $\Psi \sim \text{Normal}(m,v) $, we assumed $\Psi \sim \text{Normal}(\mu_{\psi},\sigma_{\psi}^2)$, and observed $\{Y_i\}_{i=1}^n|\Psi = \psi \sim \text{Normal}(\psi,\sigma_x^2)$, then we could recover the case introduced above via the change of coordinates $x = (y - m)/\sqrt{v}$. Apply this change of coordinates to express $\Theta$ as a function of $\Psi$. Show that the true prior for $\Theta$ is $\text{Normal}(0,1)$, and our assumed prior is $\text{Normal}((\mu_{\psi} - m)/\sqrt{v},\sigma_{\psi}^2/v)$. 

_Type your answer here, replacing this text._

So, we can assume, without loss of generality, that the true prior is $\Theta \sim \text{Normal}(0,1)$ and interpret:

1. $\mu_{\theta}$ as the standardized discrepancy in means, $ (\mu_{\psi} - m)/\sqrt{v}$, and
1. $\sigma_{\theta}$ as the ratio of the assumed and true standard deviations $\sigma_{\psi}/v$. 

## 1c) Bias and Variance in the Estimators

Repeat steps (b - f) from discussion 5 to compute the Bayes risk of the posterior mean estimator and the MLE, averaging over the *true* prior model:

$$R_B(\hat{\theta}) = \mathbb{E}_{\Theta,\{X_i\}_{i=1}^n}[(\hat{\theta}(\{X_i\}_{i=1}^n) - \Theta)^2].  $$

Note, most of this analysis will be the same. However, the posterior mean estimator will now include an additional bias associated with the misspecification error.

### 1c (i)

Find the Frequentist and Bayes bias in the MLE and posterior mean estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

_Type your answer here, replacing this text._

### 1c (ii) 

Find the Frequentist and Bayes variance in the MLE and the posterior mean estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

_Type your answer here, replacing this text._

### 1c (iii) 

Find the Bayes risk for each estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

_Type your answer here, replacing this text._

## 1d) Risk Comparison

Show that, the difference between the Bayes risks for the MLE and the posterior mean estimator is:

$$R_B(\hat{\theta}_{\text{MLE}}) - R_B(\hat{\theta}_{\text{mean}}) = \left(1 - (1 - \lambda)^2 \right) R_B(\hat{\theta}_{\text{MLE}}) - \lambda^2 (\mu_{\theta}^2 + 1), \text{ where } R_B(\hat{\theta}_{\text{MLE}}) = \frac{\sigma_x^2}{n}$$



_Type your answer here, replacing this text._

Call this difference in risks $\Delta R$. The posterior mean estimator is more reliable than the MLE as long as $\Delta R > 0$. It is much more reliable if $\Delta R \gg 0$. It is less reliable if $\Delta R < 0$. 

In the space below, write a function that can compute $\Delta R$ given the parameters:

1. The standardized error in the estimated prior mean: $\mu_{\theta} = (\mu_{\psi} - m)/\sqrt{v}$

1. The ratio of the assumed prior standard deviation and true standard deviation: $\sigma_{\theta} = \sigma_{\psi}/\sqrt{v}$

1. The standard deviation in the sampled data: $\sigma_x$

1. The sample size $n$.

Your code should be vectorized so it can compute $\Delta R$ for many different input parameter combinations efficiently.

In [ ]:
# Prepare your code here

In the space below, generate a three plots, each showing $\Delta R$ as a function of $\mu_{\theta} \in [-5,5]$ and $\sigma_{\theta} \in (0.1,10]$ for $\sigma_x = 1$ and $n = 1$, $n = 10$, and $n = 100$. Use the following display settings:

1. Set your $x$-axis to $\mu_{\theta}$ and your $y$ axis to $\sigma_{\theta}$. Use a log 10 scale for $\sigma_{\theta}$.

1. Display the surface $\Delta R(\mu,\sigma)$ as a heatmap with overlaid contours. Add a colorbar for each plot.

1. Overlay, in bold, the contour corresponding to $\Delta R(\mu,\sigma) = 0$. 

In [ ]:
# Prepare your code here

## 1e) Conclusions

Whenever you generate a plot, you should reason with the result. In the space below, describe the plots you generated above, highlighting observations you think would be important to a user who wants to adopt the posterior mean estimator, but is concerned that they will use the wrong prior parameters. Make sure that your description answers the question: "*How large can the errors in the prior parameters be before the MLE is more reliable than the posterior mean estimator?*"

_Type your answer here, replacing this text._

Suppose that you select the correct prior variance, $\sigma_{\theta}^2 = 1$. Find the range of prior means, $\mu_{\theta}$ for which the posterior mean estimator is more reliable than the MLE.

_Type your answer here, replacing this text._

Suppose that you select the correct prior mean, $\mu_{\theta} = 0$. Find the range of prior variances, $\sigma_{\theta}^2$ for which the posterior mean estimator is more reliable than the MLE.

_Type your answer here, replacing this text._

You should notice that these answers are different. Given the correct variance, there is an upper and lower bound on the assumed means outside of which the MLE is more reliable. In contrast, given the correct prior mean, the posterior mean estimator is more reliable whenever $\sigma_{\theta}^2$ is greater than a lower bound. In other words, there is no upper bound on the error in the assumed variance past which we should use the MLE. We are safe to use the posterior mean estimator *no matter* how much we overestimate the prior variance. 

Justify this observation conceptually.

_Type your answer here, replacing this text._

# Question 2: Fitting Prior Parameters

Consider the following conjugate Bayesian model:

1. $\{\Lambda_i\}_{i=1}^n$ are drawn i.i.d. from a $\text{Gamma}(\alpha,\beta)$ distribution:

    $$ \Lambda_i \geq 0, \quad f_{\Lambda_i}(\lambda) = \frac{b^a}{\Gamma(a)} \lambda^{a - 1} e^{-b \lambda} $$

1. $\{X_i|\Lambda_i = \lambda \}_{i=1}^n$ are drawn independently from a $\text{Poisson}(\lambda)$ distribution

    $$X_i \in \{0,1,2,...\}, \quad \text{Pr}(X_i = x|\Lambda_i = \lambda) = \frac{\lambda^x}{x!} e^{-\lambda}$$

You can observe the counts $\{X_i\}_{i=1}^n$, but not the rates, $\{\Lambda_i\}_{i=1}^n$. You would like to use your observations to estimate the prior parameters $\alpha, \beta$ so that, given a new observation $X_{n+1} = x_{n+1}$, you could form posterior estimates to $\Lambda_{n+1}$. 

## 2a) Posterior Inference

Suppose, for now, that you had already fit for $a$ and $b$ (see below), and we observed $X_{n+1} = x_{n+1}$. Find the posterior distribution for $\Lambda_{n+1}|X_{n+1} = x_n$ and provide a formula for the maximum a posteriori estimator for $\Lambda_n$. 

_Type your answer here, replacing this text._

## 2b) The Evidence

In a Bayesian model the evidence is the marginal distribution of the data, marginalizing over the unknowns. 

Find the marginal distribution of $X_i$, marginalizing over $\Lambda_i$. Express your answer by completing the probability mass function:

$$\text{Pr}(X_i = x_i) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~} $$

To close your integral you may use the value of the gamma integral suggested by the normalization factor of the gamma distribution:

$$\int_{\lambda = 0}^{\infty} \lambda^{c - 1} e^{-d \lambda} d\lambda = \frac{\Gamma(c)}{d^c}. $$


_Type your answer here, replacing this text._

This distribution is also called the *prior predictive distribution* since it is the distribution we would use to predict the value of a new data point before observing any other data. Notice that its parameters are the parameters of the prior.

So, prior predictive distribution, evaluated at an observed outcome, is, as a function of the prior parameters, the likelihood function for the prior parameters. It follows that, selecting the prior parameters that maximize the evidence is equivalent to selecting the maximum likelihood *prior* parameters. The prior parameters that maximize the evidence are the prior parameters that, if true, would make the observed data most likely!

Note: this is why we call the prior predictive distribution the evidence. The larger the prior predictive distribution, evaluated at the observation, the more evidence we have in favor of our prior model (here, our choice of prior parameters. )

In the space below, complete the expressions:

$$\text{Pr}(X_1 = x_1,X_2 = x_2,...,X_n = x_n) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~}, \quad  \log(\text{Pr}(X_1 = x_1,X_2 = x_2,...,X_n = x_n)) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~}$$

_Type your answer here, replacing this text._

# Question 3: Approximate Inference via Sampling

This question picks up where you left off in lab 4. For details on the model, review question 1 from [lab 4](https://data102.datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fds-102%2Ffa26-materials&urlpath=tree%2Ffa26-materials%2Flab%2Flab04%2Flab04.ipynb&branch=main).

As we introduce more complexity to a model, the **conjugacy property quickly breaks and we have to resort to approximate inference**. In this class, we'll focus primarily on *sampling* for approximate inference: this will be the topic of the next few lectures and next week's labs. In sampling-based approaches, **we don't even try to get the exact posterior: instead, we generate a bunch of samples from it, and use those to approximate the distribution.**

In this question you will get a taste for probabilistic programming using `PyMC`. Spend some time perusing the [documentation](https://www.pymc.io/projects/docs/en/stable/learn/core_notebooks/pymc_overview.html), but don't worry if there are parts that don't make sense yet. The Quickstart guide is a useful starting point. 

We'll be using PyMC to run an algorithm called Markov Chain Monte Carlo (MCMC), which you'll learn about this week and next. We'll start by using the same model from `Question 1`, and compare the results from MCMC with the exact solutions we calculated above. Then, we'll add an extra parameter to the model and make things more complex: even though we can no longer compute our posterior in closed form, MCMC will still generate samples that we can use to estimate each $\theta_i$.


In [ ]:
# Create a dummy model so that one-time initialization
# happens while you're reading over the code in the next cell.

# Note: this and the following cells may take a while to run

# You can ignore the output of this cell.

with pm.Model() as model:
    dummy = pm.Beta('dummy', alpha=1, beta=1)
    pm.sample(1, return_inferencedata=False, progressbar=False)

## 2a) Building and sampling from a PyMC model

First, let's get acquainted with the structure of PyMC, and show that methods of approximate inference are (in many cases) just as good as having access to the exact distribution. Just like in `Question 1`, we're still trying to estimate the SAR using the posterior mean; only this time, we'll be using PyMC to get an approximate answer!

#### So, how does PyMC work? What does it even do?

In essence, PyMC is just a number generator with extra steps. Given a *likelihood* and a *prior*, it simulates thousands of samples to create an empirical distribution that approximates the true posterior distribution. With those samples, we're able to calculate values of interest (such as the posterior mean, MAP, etc.), and create estimates of $\theta$.

#### How do we tell PyMC what our model looks like?

As in any Bayesian parameter estimation problem, in order to find the posterior distribution, we must supply a **likelihood** and **prior**. To do this, we can instantiate [`Distribution`s](https://www.pymc.io/projects/docs/en/v3/Probability_Distributions.html) and their associated parameters to represent the random variables in our problem (you can take a look [here](https://www.pymc.io/projects/docs/en/v3/api/distributions.html) for a list of supported distributions).

Once defined, these distribution objects are like any random variable in statistics; you can add, subtract, multiply, divide, and use mathematic operators on them (though sometimes you may need to use PyMC's own math operators). They can also function as either likelihoods or priors, depending on the parameters you feed them:
1. If a `Distribution` object's parameters **do not** depend on other `Distribution` objects, these represent **priors**.
2. If a `Distribution` object's parameters **do** depend on other `Distribution` objects, these represent **likelihoods**.

#### What about our data?

Now that we've specified a likelihood and prior, we have to consider our *data*; remember, the posterior describes the distribution of $\theta$ given the data we observed! To do this, the `Distribution` class has a nifty little argument that we can pass in called `observed`. Using the `observed` argument, we can pass in our observed data to the random variable/`Distribution` object that corresponds to our observations, and let PyMC handle the rest!

#### Now, try it out for yourself!

Take a look at the cell below, and **define the theta and X nodes to approximate the posterior distribution.** The PyMC website has documentation for defining variables according to the Beta and Binomial distribution. If you have no idea how to proceed, the PyMC Quickstart guide is a great place to start!

In [ ]:
# TODO: complete the function
def approximate_inference_MCMC(
    alpha_value, beta_value, study_df = study_df
):
    """
    Creates and generates samples from a PyMC model of
    the posterior distribution that corresponds to the
    graphical model in Q.1, using Markov Chain Monte Carlo (MCMC)
    
    Inputs:
        alpha_value, beta_value : floats, parameters of
        the prior Beta Distribution
        study_df : DataFrame containing study data
    
    Outputs: (model, trace)
        model is a PyMC model object, which represents the graphical model
        trace is a PyMC trace object, which represents 2000 samples
            of everything from the posterior
    """
    # Defines the graphical model
    with pm.Model() as model:
        # The prior for theta is a Beta distribution with parameters
        # alpha and beta, and there's one for each study.
        # Make sure to name this parameter 'theta' so so this lab can reference it later. 
        theta = ...
        
        # The likelihood for X is binomial, with parameter p=theta,
        # observed counts in study_df['X'], and observed N similarly
        X = ...
        
        # Generate samples from the posterior distribution using : run 4
        # Markov chains of sampling in parallel, generating 500 samples
        # each.
        trace = pm.sample(500, chains=4, tune=1000, target_accept=0.95, return_inferencedata=False, progressbar=False)
    
    return (model, trace)

In [ ]:
grader.check("q2a_i")

Now, when we call the `approximate_inference_MCMC` function, we run the sampler and pass in fixed values of the hyperparameters alpha and beta.

In [ ]:
# Run approximate inference
model, trace = approximate_inference_MCMC(10, 20)

# Get posterior samples of theta
thetas = trace['theta']
thetas
print(thetas.shape)

<!-- BEGIN QUESTION -->

Note that the shape of ``thetas`` is (N x M).  What are N and M, and what does each mean?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 3b) Using the output of PyMC

Now that we've run our sampler, we now have access to the posterior distributions of *all* the random variables we defined in PyMC. Using these empirical distributions, we can now calculate the posterior means for each $\theta_i$. But before we do that, let's visualize the samples we got back.

Generate a histogram of all 2,000 posterior samples for $\theta_2$ (the SAR for Study 2). Use the `sns.histplot` function with `stat='density'`.

In [ ]:
# TODO: Create histogram of posterior samples
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

How do the samples compare to the two different estimates you saw in Question 1?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## 3c) Compute Posterior Mean Estimates from Samples

Fill in the function that computes posterior mean estimates for each $\theta_i$ for different parameters $\alpha, \beta$ of the prior distribution.

In [ ]:
def empirical_posterior_mean_estimates(alpha_value, beta_value, study_df = study_df):
    """ 
    Computes posterior mean estimates of theta_i by performing approximate inference
    and then sampling from the posterior distribution:
    
    Inputs:
        alpha_value, beta_value : floats, parameters of the prior Beta Distribution
        study_df : DataFrame containing study data
        
    Output:
        posterior_estimates : (num_studies,) 1-D array of the same length as the 
            number of studies. posterior_estimates[i] contains the 
            mean estimate for theta_i based on running MCMC
    
    """
    ...
    posterior_estimates = ...
    return posterior_estimates

In [ ]:
grader.check("q2c")

## 3d) Comparing Results from Question 1 and Question 2

Now that we've utilized conjugacy to get the theoretical distribution of the exact posterior (`Question 1`) and applied Markov Chain Monte Carlo to get an approximate empirical distribution (`Question 2`), let's compare the two to see how good our approximation was!

Below, we've provided code to make a 4x3 plot such that each subplot corresponds to a study. 

Each subplot contains 2 curves and a frequency histogram:
- The PDF of the prior distribution of $\theta_i$
- The PDF of the true posterior distribution $\theta_i|X_i$ computed in closed form, as in Q.1
- The histogram of posterior samples of $\theta_i|X_i$ computed in Q.2

Take a look through the subplots, and compare the fit between our empirical and theoretical distributions.

In [ ]:
def plot_densities(alpha_value, beta_value, study_df = study_df): 
    """
    Plots for each study the prior distribution, true posterior,
    and histogram of posterior samples using MCMC
    
    Inputs:
        alpha_value, beta_value : floats, parameters of the prior Beta Distribution
        study_df : DataFrame containing study data
        
    Outputs:
        fig : Figure with 12 subplots
    """
    fig, axs = plt.subplots(4, 3)
    fig.set_figheight(15)
    fig.set_figwidth(15)

    
    theta = np.arange(0, 1.01, 0.01)
    prior = beta.pdf(theta, alpha_value, beta_value)
    
    model, trace = approximate_inference_MCMC(alpha_value, beta_value, study_df) 
    samples = trace['theta'] 
    
    for i in range(4):
        for j in range(3):
            idx = 3*i+ j
            X_i = study_df.loc[idx, 'X']
            N_i = study_df.loc[idx, 'N']
            study_name = f'Study {idx}'
            true_posterior = beta.pdf(theta, alpha_value+X_i, beta_value+N_i-X_i) 
            
            ax = axs[i, j]
            ax.plot(theta, prior, label = 'Prior')
            ax.plot(theta, true_posterior, label = "Theoretical Posterior")
            ax.hist(samples[:,idx], label = "Empirical Posterior", density=True, alpha = 0.7)
            ax.set_title(study_name)
            ax.legend()
    
    plt.tight_layout()        
    plt.show()
    return fig

In [ ]:
# Plot the resulting densities for a weak prior
fig1 = plot_densities(2, 4, study_df = study_df)

In [ ]:
# Plot the resulting densities for a strong prior
fig2 = plot_densities(20, 40, study_df = study_df)

<!-- BEGIN QUESTION -->

### 3d (i) 

Compare the curve of the theoretical distribution with the histogram of samples from the empirical posterior. Are they similar or different?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 3d (ii)

Compare the two figures corresponding to 'weak' prior $\theta_i \sim Beta(2,4)$ and 'strong' prior  $\theta_i \sim Beta(20,40)$. How are they different? Explain why.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## 3e) Approximate Inference for a More Complex Model

The previous 2 parts served as a sanity check that the approximate inference techniques used by PyMC can approximate the theoretical posterior. The usefulness of such packages becomes apparent when we are dealing with more complex models that don't have conjugacy properties.

Consider the following graphical model:

![](model_asymptomatic.png)

Recent studies have shown that a large fraction of COVID cases do not show symptoms, but all of the studies considered here tested only symptomatic cases. This means that the probability of testing positive (which what we observe) isn't the same as the SAR $\theta_i$! 

The estimates of the asymptomatic rate fall in the range $[0.18, 0.43]$. We assume a prior $A\sim Uniform(0.18, 0.43)$. This means that the probability that a person in a study tests positive is really $\theta_i*(1-A)$. Hence:

$$X_i|\theta_i, A \sim Binomial(N_i, \theta_i\cdot (1 - A))$$

#### Complete the `approximate_inference_asympotmatic_MCMC` function to add dependence on the asymptomatic rate:

In [ ]:
# TODO: complete the function
def approximate_inference_asympotmatic_MCMC(alpha_value, beta_value, study_df = study_df):
    """
    Creates and fits a PyMC model corresponding to the graphical model above
    
    Inputs:
        alpha_value, beta_value : floats, parameters of the prior Beta Distribution
        study_df : DataFrame containing study data
    
    Outputs: (model, trace)
    """
    with pm.Model() as model:
        theta = ...
        A = ...
        X = ...
        
        trace = pm.sample(500, tune=1000, target_accept=0.95, return_inferencedata=False, progressbar=False)
    return (model, trace)

In [ ]:
grader.check("q2e_i")

<!-- BEGIN QUESTION -->

Notice that the `trace` now contains samples for both `theta` and `A`!

Plot a histogram of the posterior estimates for $A$ if $\alpha=5$ and $\beta=10$.

In [ ]:
model, trace = approximate_inference_asympotmatic_MCMC(5, 10)
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Assuming the model we defined is correct, what can you conclude about the asymptomatic rate $A$ based on the studies and the model?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Well done!
You've reached the end of the lab! Make sure you double check your work and make sure that you've answered all the written portions of the lab.

Before you submit to Pensieve, make sure you pass all the autograded portions of this lab. **Run the last cell to generate a zip file of your lab submission.**

To submit your lab to Pensieve, submit the zip file to Lab 4 Code.

In [ ]:
import matplotlib.image as mpimg

img = mpimg.imread('baby_rock_hyrax.jpg')
imgplot = plt.imshow(img)
imgplot.axes.get_xaxis().set_visible(False)
imgplot.axes.get_yaxis().set_visible(False)
plt.show()

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

In [ ]:
grader.export(pdf=False, force_save=True)